# 🧭 15 · Image Registration
### Mapping one image's points onto another's — rigid, affine and deformable transforms, and how to find them

**Part G · Across time and space** · [⟵ 14 Object tracking](14_object_tracking.ipynb) · [🏠 Course home](../README.md) · [16 Denoising & super-resolution ⟶](16_denoising_and_super_resolution.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/15_image_registration.ipynb)

| | |
|---|---|
| **Hardware target** | CPU, a laptop is enough (grid searches take ~10–20 s total) |
| **Core frameworks** | NumPy, SciPy (`scipy.ndimage.affine_transform`), Matplotlib |
| **Prerequisites** | Module 04 (frequency domain), Module 05 (keypoints), Module 06 (edges), Module 09 (dimensionality — for the SVD used by Kabsch) |
| **Estimated time** | 70–90 minutes |
| **02-740 topics covered** | registration taxonomy, rigid/affine/projective/curved transforms, the registration pipeline, similarity metrics (SSD, correlation, mutual information), point-set and intensity-based registration, regularisation and optimisation |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All images and point sets in this notebook are SIMULATED by code — nothing is downloaded.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- define image registration as finding a spatial transform that maps points in one image to corresponding points in another, and place a given registration problem in its taxonomy (2D/3D, rigid vs. non-rigid, mono- vs. multi-modal, extrinsic vs. intrinsic);
- name the four classic steps of a registration pipeline (feature detection → feature matching → transform estimation → resampling) and what each one does;
- compute and compare three similarity metrics — sum of squared differences, normalised cross-correlation, and mutual information — and explain *why* only the last one is reliable for multi-modal images;
- implement point-set registration with the **Kabsch algorithm** and recover a known rigid transform from noisy landmark correspondences;
- implement a simple **intensity-based** registration by direct search, and demonstrate concretely why correlation-based matching can fail under a modality change while mutual information does not.

## 🗺️ Contents

1. What is image registration, and how is it classified?
2. Four reasons to register images
3. The registration pipeline: feature detection and similarity metrics
4. Point-based registration: the Kabsch algorithm
5. Intensity-based registration: direct search, and why mutual information matters
6. Regularisation, optimisation, and a glance at non-rigid registration

## 🧫 The biological question

No single image tells the whole story. A confocal stack taken today cannot be compared to one taken tomorrow unless both are expressed in the same coordinate frame; a fluorescence image and an electron-microscopy image of the *same* cell can only be fused into one picture of where a protein sits relative to an organelle if the two are first aligned; and tracking a moving cell (Module 14) is itself a registration problem in disguise — "where did everything that didn't move stay, and where did the one thing that moved go?" **Image registration** is the general tool underneath all of these: a spatial transformation that maps points in a *moving* image onto their corresponding points in a *fixed* (reference) image, so that information from both can be interpreted in one shared coordinate system.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 15
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

We reuse `simulate_nuclei_image` and `simulate_registration_pair` (a known-ground-truth rigid transform between a fixed and moving image), `estimate_rigid_transform` (Kabsch point-set registration), `mutual_information`, and `pearson_colocalization` (which, applied to two whole images rather than two channels, is exactly normalised cross-correlation).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def simulate_registration_pair(sim: dict, angle_deg: float = 12.0, translation: tuple = (8.0, -5.0),
                                scale: float = 1.0, seed: int = 0):
    """Make a SIMULATED 'moving' image by applying a known rigid/similarity transform to ``sim['image']``.

    Returns ``fixed``, ``moving`` and ``true_params`` (angle in degrees, translation (dy, dx), scale) —
    the ground truth that a registration algorithm should recover.
    """
    fixed = sim["image"]
    theta = np.deg2rad(angle_deg)
    matrix = scale * np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    center = np.array(fixed.shape) / 2
    offset = center - matrix @ center - np.array(translation)
    moving = ndi.affine_transform(fixed, matrix, offset=offset, order=1, mode="constant", cval=fixed.min())
    rng = np.random.default_rng(seed)
    moving = add_gaussian_noise(moving, 0.01, rng).clip(0, None)
    return dict(fixed=fixed, moving=moving,
                true_params=dict(angle_deg=angle_deg, translation=translation, scale=scale),
                source="SIMULATED")


def estimate_rigid_transform(src_pts: np.ndarray, dst_pts: np.ndarray):
    """Least-squares rigid (rotation + translation) transform that best maps ``src_pts`` onto
    ``dst_pts`` (point-set registration via the Kabsch algorithm). Returns (R, t) such that
    dst ≈ src @ R.T + t."""
    src_mean, dst_mean = src_pts.mean(axis=0), dst_pts.mean(axis=0)
    src_c, dst_c = src_pts - src_mean, dst_pts - dst_mean
    H = src_c.T @ dst_c
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, d]) @ U.T
    t = dst_mean - R @ src_mean
    return R, t


def mutual_information(image_a: np.ndarray, image_b: np.ndarray, bins: int = 32) -> float:
    """Mutual information between two co-registered images, estimated from their 2-D joint
    intensity histogram — the similarity metric of choice for multi-modal registration, where
    intensities are related but not linearly (unlike single-modality correlation)."""
    hist2d, _, _ = np.histogram2d(image_a.ravel(), image_b.ravel(), bins=bins)
    pxy = hist2d / hist2d.sum()
    px, py = pxy.sum(axis=1), pxy.sum(axis=0)
    px_py = px[:, None] * py[None, :]
    nz = pxy > 0
    return float(np.sum(pxy[nz] * np.log(pxy[nz] / px_py[nz])))


def pearson_colocalization(ch_a: np.ndarray, ch_b: np.ndarray, mask: np.ndarray | None = None) -> float:
    """Pearson's colocalization coefficient: the ordinary linear correlation between two
    channels' pixel intensities, restricted to ``mask`` if given (e.g. just the cell interior)."""
    a = ch_a[mask] if mask is not None else ch_a.ravel()
    b = ch_b[mask] if mask is not None else ch_b.ravel()
    a, b = a - a.mean(), b - b.mean()
    denom = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / denom) if denom > 0 else 0.0


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · What is image registration, and how is it classified?

Formally, registration seeks a mapping function $T$ such that for every point $P$ in one image, $Q = T(P; a)$ gives the corresponding point in the other image, where $a$ are the transform's parameters. The same taxonomy recurs across every registration paper and tool you will meet:

- **Dimensionality** — 2D–2D (e.g. aligning two microscopy fields), 3D–3D (two tomographic volumes), 2D–3D (a single slice onto a volume), or time-series (a movie, Module 14's setting).
- **Nature of the registration basis** — *extrinsic* (relies on artificial markers deliberately introduced into the sample, e.g. gold fiducial beads in cryo-electron tomography — easy to detect, but requires sample preparation), *intrinsic* (relies purely on the image content itself — harder, but needs no special markers), or *non-image-based* (follows a known imaging geometry directly, e.g. the fixed rotation schedule of a tomographic tilt series).
- **Domain of the transformation** — *global* (one transform applies to the whole image) or *local* (different regions get their own transform).
- **Modalities involved** — *mono-modal* (both images are, say, fluorescence micrographs) or *multi-modal* (one fluorescence, one electron-microscopy image of the same sample — genuinely useful because no single modality tells the whole story, but harder because the two images' intensities are related in a complicated, non-linear way).
- **Nature of the transformation** — rigid, affine, projective, or curved (free-form); the next cell makes each of these concrete.

The transform families, from most to least constrained, all share the form $Q = T(P; a)$:

$$\text{rigid: } Q = Rv + t \quad (R^TR = I) \qquad\qquad \text{affine: } Q = Av + t \quad (A \text{ any } 2\times2)$$

$$\text{projective: } \rho\, Q = \begin{pmatrix}A & B & C\\ D & E & F\\ G & H & 1\end{pmatrix} v \qquad\qquad \text{curved: } Q = T_{\text{global}}(v) + T_{\text{local}}(v)$$

*In words:* a **rigid** transform only rotates and translates — every distance and angle in the image is preserved, which is exactly right when the object itself did not deform (e.g. a rigid stage drift between two acquisitions of the same fixed sample). An **affine** transform additionally allows shear and non-uniform scaling but still keeps straight lines straight and parallel lines parallel. A **projective** transform is the most general *linear* mapping (it can make parallel lines converge, as in a camera's perspective). A **curved** (free-form, deformable) transform is needed whenever different parts of the image must warp independently — the regime where a single global rigid or affine transform cannot possibly fit, such as registering two different individuals' anatomy to a common atlas.

## 2 · Four reasons to register images

1. **From different angles** — e.g. tomography: a series of 2D projections taken at different tilt angles is registered and reconstructed into one 3D volume.
2. **At different times** — e.g. cell tracking (Module 14): registering frame $t$ to frame $t+1$ reveals exactly how far, and in which direction, something moved.
3. **Using different devices** — e.g. fusing a light-microscopy image with an electron-microscopy image of the same sample, or a PET scan with a CT scan: each modality alone is an incomplete picture, but no single image can describe a complex object comprehensively, so registering them together recovers a more complete story.
4. **From different individuals** — e.g. building a brain or tissue atlas: images from many subjects are registered onto one reference template so that corresponding anatomical structures end up in the same place, making cross-subject comparison possible (almost always needing a non-rigid/deformable final step, since no two individuals' anatomy is related by a single rigid or affine map).

In [ ]:
# ── 2.1 Build a fixed/moving pair with a KNOWN ground-truth rigid transform ──
sim = simulate_nuclei_image(n_cells=20, size=160, seed=SEED)
pair = simulate_registration_pair(sim, angle_deg=12.0, translation=(8.0, -5.0), seed=SEED)
fixed, moving, true_params = pair["fixed"], pair["moving"], pair["true_params"]
print("True transform used to create 'moving' from 'fixed':", true_params)

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
show_image(axes[0], fixed, "fixed")
show_image(axes[1], moving, "moving (rotated + translated)")
show_image(axes[2], fixed - moving, "naive difference (misaligned)")
fig.tight_layout()

🔍 **What just happened?** `fixed` and `moving` show the *same* simulated cells, but `moving` has been rotated by 12° and translated by (8, −5) pixels from `fixed` — exactly the kind of stage drift or re-mounting offset you'd see between two real acquisitions. The naive pixelwise difference is dominated by this misalignment, not by any real biological difference — which is precisely why registration has to happen *before* any meaningful comparison between two images.

## 3 · The registration pipeline: feature detection and similarity metrics

A registration pipeline has four conceptual steps: **(1) feature detection** (find distinctive structure to anchor the alignment on — landmarks, intensity patterns, or feature points such as Module 05's keypoints), **(2) feature matching** (define a similarity measure between features or whole images and use it as an optimisation objective), **(3) transform-model estimation** (search over the transform's parameters to optimise that similarity, usually with regularisation — Section 6), and **(4) image resampling** (once the transform parameters are known, resample the moving image onto the fixed grid, which requires interpolation since the transformed sampling grid will not land exactly on integer pixel coordinates).

Three similarity metrics recur constantly for step (2):

- **Sum of squared differences (SSD):** $\tfrac12 \sum (I(x) - J(T(x)))^2$. Simple and fast, and perfectly fine in the simplest case — but it implicitly assumes the two images' intensities are *directly comparable* (the same object looks the same brightness in both), which fails the moment the two images come from different modalities or even just different illumination.
- **Normalised cross-correlation (NCC):** captures a *linear* relationship between the two images' intensities, so it tolerates a uniform brightness/contrast rescaling that SSD cannot — but it is still blind to anything beyond a straight-line intensity relationship.
- **Mutual information (MI):** measured from the two images' *joint histogram*, MI asks "how much does the intensity in image A tell me about the intensity in image B at the same location?" without assuming the relationship between them is linear (or even monotonic) at all. This is exactly what multi-modal registration needs: a T1- and a T2-weighted MRI of the same brain have utterly different absolute intensities for the same tissue, so SSD and NCC are unreliable there, while MI — which only cares whether the joint histogram concentrates into a small number of (possibly nonlinear) clusters at the correct alignment — is not.

In [ ]:
# ── 3.1 Compare SSD / NCC / MI at the correct vs. a wrong alignment ──
def ssd(a, b):
    return float(0.5 * np.sum((a - b) ** 2))

wrong_shift = np.roll(np.roll(moving, 20, axis=0), -15, axis=1)  # an arbitrary wrong alignment

for name, candidate in [("correctly aligned (fixed vs. fixed)", fixed),
                        ("moving, uncorrected", moving),
                        ("an arbitrary wrong shift", wrong_shift)]:
    print(f"{name:38s}  SSD={ssd(fixed, candidate):9.2f}   "
          f"NCC={pearson_colocalization(fixed, candidate):+.3f}   "
          f"MI={mutual_information(fixed, candidate):.3f}")

🔍 **What just happened?** Against itself, `fixed` has SSD = 0, NCC = 1, MI at its maximum — the extremum every similarity measure should hit at perfect alignment. The uncorrected `moving` and the arbitrary wrong shift both score far worse on all three metrics here, because this is a **mono-modal** comparison (both images are the same simulated fluorescence channel) — exactly the case where SSD and NCC are perfectly adequate. Section 5 constructs a multi-modal pair where that stops being true.

✏️ **Your turn.** `np.roll` wraps pixels around the image border, which is not physically realistic but is a convenient stand-in for "some other misalignment." Try a few different roll amounts and confirm that all three metrics agree on which shift is closer to correct — they should, as long as both images are the same modality.

## 4 · Point-based registration: the Kabsch algorithm

When distinctive **landmarks** can be identified in both images — the same few nuclei centroids, the same fiducial markers, the same anatomical points — registration reduces to a much smaller, closed-form problem: given $n$ corresponding point pairs $\{(p_i, q_i)\}$, find the rigid transform $R, t$ that best maps $p_i \to q_i$ in a least-squares sense. The classic landmark-based recipe is: (1) compute each point set's centroid and subtract it off (this handles the translation), (2) find the rotation that minimises the sum of squared distances between the recentred, corresponding points — this is the **Kabsch algorithm**, solved in closed form via the SVD of the cross-covariance matrix of the two centred point sets (the same SVD machinery as Module 09's PCA).

In [ ]:
# ── 4.1 Recover a KNOWN rigid transform from noisy landmark correspondences ──
n_landmarks = 15
true_angle_deg, true_translation = 17.0, (6.0, -9.0)
theta = np.deg2rad(true_angle_deg)
R_true = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
t_true = np.array(true_translation)

src_points = rng.uniform(10, 150, size=(n_landmarks, 2))          # e.g. nuclei centroids in image A
dst_points = src_points @ R_true.T + t_true                        # the same nuclei, found in image B
dst_points_noisy = dst_points + rng.normal(0, 0.6, size=dst_points.shape)  # realistic localisation noise

R_est, t_est = estimate_rigid_transform(src_points, dst_points_noisy)
recovered_angle = np.degrees(np.arctan2(R_est[1, 0], R_est[0, 0]))
print(f"True  rotation {true_angle_deg:.1f}°, translation {true_translation}")
print(f"Kabsch rotation {recovered_angle:.2f}°, translation ({t_est[0]:.2f}, {t_est[1]:.2f})")

residual = np.linalg.norm(src_points @ R_est.T + t_est - dst_points_noisy, axis=1)
print(f"Mean residual after registration: {residual.mean():.3f} px (noise floor was 0.6 px)")

🧪 **Ground-truth check.** We generated `dst_points_noisy` with a *known* rotation and translation plus small Gaussian localisation noise. The Kabsch-recovered rotation and translation should land within a fraction of a degree / pixel of the truth, and the mean residual after applying the recovered transform should be close to the noise floor we injected (0.6 px) — if Kabsch were not finding the right answer, the residual would be much larger.

In [ ]:
# ── 4.2 Outliers break point-based registration — this is exactly what RANSAC is for ──
dst_with_outliers = dst_points_noisy.copy()
outlier_idx = rng.choice(n_landmarks, size=3, replace=False)
dst_with_outliers[outlier_idx] += rng.uniform(-40, 40, size=(3, 2))  # 3 badly mismatched landmarks

R_bad, t_bad = estimate_rigid_transform(src_points, dst_with_outliers)
bad_angle = np.degrees(np.arctan2(R_bad[1, 0], R_bad[0, 0]))
print(f"With 3/{n_landmarks} outlier correspondences, Kabsch recovers {bad_angle:.2f}° "
      f"(should be {true_angle_deg:.1f}°)")

🔍 **What just happened?** Just 3 badly-mismatched correspondences out of 15 are enough to visibly bias the least-squares Kabsch fit, because *every* point contributes to the sum being minimised — a few bad matches can pull the whole solution away from the truth. **RANSAC** (random sample consensus) is the standard fix: repeatedly fit the model to a small random subset of correspondences, count how many of the *remaining* points agree with that fit within some tolerance (the "consensus set"), and keep the fit with the largest consensus set — so a handful of outlier correspondences can no longer dominate the result, as long as the majority of matches are correct.

### 📘 Lecture notes — landmark-based registration in practice

In cryo-electron tomography, where sample contrast is often too low to align tilt images from content alone, gold fiducial beads are deliberately mixed into the sample purely to serve as extrinsic landmarks for exactly this kind of point-based registration — trading a small amount of sample preparation complexity for a much more reliable alignment than intensity-based registration could achieve on the raw, low-contrast projections.

## 5 · Intensity-based registration: direct search, and why mutual information matters

When no reliable landmarks are available, registration must work directly on pixel intensities: propose a candidate transform, resample the moving image with it, score the similarity against the fixed image, and repeat — search over transform parameters to **maximise** that similarity. We demonstrate this with a small brute-force grid search (in practice, gradient-based or other iterative optimisers replace the grid — Section 6).

In [ ]:
# ── 5.1 A small helper to apply a candidate rigid transform (mirrors the toolkit's convention) ──
def warp_rigid(image, angle_deg, translation, scale=1.0):
    theta = np.deg2rad(angle_deg)
    matrix = scale * np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    center = np.array(image.shape) / 2
    offset = center - matrix @ center - np.array(translation)
    return ndi.affine_transform(image, matrix, offset=offset, order=1, mode="constant", cval=image.min())

def grid_search_register(reference, target, metric_fn, angle_range=(-20, 20, 2),
                          shift_range=(-12, 12, 4)):
    """Search over (angle, dy, dx) for the warp of `reference` that best matches `target`
    under `metric_fn` (HIGHER must mean better). Returns (best_angle, best_dy, best_dx, best_score)."""
    best = (None, None, None, -np.inf)
    for angle in np.arange(*angle_range):
        for dy in np.arange(*shift_range):
            for dx in np.arange(*shift_range):
                candidate = warp_rigid(reference, angle, (dy, dx))
                score = metric_fn(candidate, target)
                if score > best[-1]:
                    best = (angle, dy, dx, score)
    return best

# Mono-modal: search for the transform of `fixed` that best matches `moving`, using NCC
best_angle, best_dy, best_dx, best_ncc = grid_search_register(
    fixed, moving, pearson_colocalization)
print(f"True transform:      {true_params}")
print(f"NCC-recovered (mono-modal): angle={best_angle:.1f}, translation=({best_dy:.1f}, {best_dx:.1f})")

🧪 **Ground-truth check.** The NCC-based grid search should land close to the true `angle_deg=12.0, translation=(8.0, -5.0)` used to build `moving` — both images are the same modality, so correlation is a perfectly reliable objective here.

In [ ]:
# ── 5.2 Now make it genuinely multi-modal: simulate a contrast-INVERTED second channel ──
# e.g. a brightfield / fluorescence pair where bright structure in one is dark in the other
moving_inverted = moving.max() - moving

best_mm_ncc = grid_search_register(fixed, moving_inverted, pearson_colocalization)
best_mm_mi = grid_search_register(fixed, moving_inverted, mutual_information)
print(f"True transform:                         {true_params}")
print(f"NCC-recovered (contrast-inverted pair):  angle={best_mm_ncc[0]:.1f}, "
      f"translation=({best_mm_ncc[1]:.1f}, {best_mm_ncc[2]:.1f})  [score={best_mm_ncc[3]:+.3f}]")
print(f"MI-recovered  (contrast-inverted pair):  angle={best_mm_mi[0]:.1f}, "
      f"translation=({best_mm_mi[1]:.1f}, {best_mm_mi[2]:.1f})  [score={best_mm_mi[3]:.3f}]")

at_truth_ncc = pearson_colocalization(
    warp_rigid(fixed, true_params["angle_deg"], true_params["translation"]), moving_inverted)
print(f"(for reference: NCC AT the true transform is {at_truth_ncc:+.3f} — strongly NEGATIVE, "
      f"so naively maximising NCC can never find it)")

🔍 **What just happened?** Inverting the moving image's contrast is a simple stand-in for "a different imaging modality": the same structures are still there, but bright and dark are swapped, so the intensity relationship between the two images is no longer "similar values at corresponding points" — it is "opposite values." At the *true* alignment, NCC is now strongly **negative**, not positive, so an algorithm that naively searches for the **maximum** of NCC walks straight past the correct answer (it finds whatever candidate happens to have the least-negative-or-most-positive correlation, which is not the true alignment). Mutual information has no such blind spot: it only measures how concentrated the *joint histogram* is, regardless of whether that concentration follows an increasing, decreasing, or any other relationship — so it recovers the true transform in both the mono-modal and the contrast-inverted case. This is exactly the robustness that made MI the standard similarity metric for genuinely multi-modal registration (e.g. CT–MRI, or light microscopy–electron microscopy).

✏️ **Your turn.** Besides simple inversion, try a more drastic nonlinear "modality change," e.g. `moving_sq = (moving / moving.max()) ** 2 * moving.max()` (a non-monotonic-feeling but still smooth remap). Does MI still find the correct transform? Does NCC do any better or worse than with plain inversion?

One more practical note, already foreshadowed in Module 14: **image stitching** — tiling many overlapping fields of view into one large mosaic, as microscopes routinely do when a sample is bigger than one field of view — is really the *same* registration problem applied pairwise to adjacent tiles, usually with a translation-only (or similarity) transform and a strong expectation of a large, correct-answer overlap region to anchor the search; the tools differ mainly in how they exploit that structure for speed, not in the underlying math.

## 6 · Regularisation, optimisation, and a glance at non-rigid registration

Minimising a similarity measure alone is not always enough: with too few constraints relative to the number of free parameters, the problem can be **ill-posed**, and even when it is well-posed, an unregularised optimisation landscape can have many local minima that trap a naive optimiser. The standard fix is to add a **regularisation term** $E_R(T)$ to the objective, $\arg\min_T \sum_k E_D(I(x_k), J(T(x_k))) + \lambda E_R(T)$, which encodes prior knowledge (e.g. "the true deformation should be smooth") and makes the optimisation both well-posed and better-behaved.

In practice, the transform parameters are updated iteratively rather than searched by brute force: **gradient-based** methods (steepest descent, Newton's method, and their variants) take a derivative of the similarity measure with respect to the transform parameters and step in the improving direction — fast when the objective is smooth, but exactly where SSD's instability under illumination/modality changes (the derivative can point in a completely wrong direction) becomes a real problem, and where MI's smoother behaviour pays off again. **Gradient-free** methods are used when the objective is not easily differentiable.

Everything above has used a single **global**, **rigid** transform. When different parts of an anatomy move or deform independently — the regime that needs a **non-rigid, local** transform — a common approach is **free-form deformation with B-splines**: the total transform is written as the sum of a global part and a local part, $T(x) = T_{\text{global}}(x) + T_{\text{local}}(x)$, where $T_{\text{local}}$ is a smooth function controlled by a modest number of *control points* spread over the image; moving a control point smoothly deforms only the nearby image region, and more control points buy more local precision at the cost of more parameters to estimate. We do not implement B-spline registration here (it is a genuinely more advanced numerical-optimisation topic), but you should recognise it by name as the standard way to go beyond a single global rigid or affine transform — it is exactly the tool used, for example, to register brain images across different individuals (Section 2's fourth scenario).

### 📘 Lecture notes — why registration always needs *some* optimisation strategy

Every registration method in this notebook — closed-form Kabsch, brute-force intensity search, or (in real tools) gradient-based iterative refinement — is solving the same optimisation problem from Section 3: choose transform parameters that make a similarity measure extremal. The differences between methods are really differences in *how much structure* is assumed (closed-form solutions exist only for specific, constrained problems like rigid point-set alignment) and *how the search is organised* (brute force vs. gradient descent vs. RANSAC-style robust fitting) — not a difference in what is fundamentally being optimised.

## 📌 Key takeaways

- Registration finds a transform $Q = T(P; a)$ mapping points in one image onto corresponding points in another; the transform family (rigid → affine → projective → curved) should match how much the underlying structure actually deformed.
- The classic pipeline is feature detection → feature matching (a similarity metric) → transform estimation (with regularisation) → resampling.
- **SSD** and **NCC** are simple and effective for mono-modal registration but assume an increasingly specific (identity, then linear) intensity relationship between the two images.
- **Mutual information**, computed from the joint intensity histogram, makes no linearity assumption and is the standard choice for multi-modal registration, where SSD and NCC can actively point an optimiser toward the wrong answer.
- **Point-based (Kabsch) registration** gives an exact, closed-form rigid-transform estimate from corresponding landmarks, but a handful of mismatched correspondences can bias it badly — motivating RANSAC-style robust fitting.
- A single global rigid/affine transform is not always enough; **non-rigid, free-form (B-spline) deformation** is the standard way to let different parts of an image deform independently, at the cost of more parameters to estimate.

## ✅ Checkpoint

1. Why is mutual information preferred over SSD for registering a fluorescence image to an electron-microscopy image of the same sample?
   <details><summary>Answer</summary>SSD (and NCC) assume the two images' intensities are related identically or linearly; a fluorescence and an EM image of the same structure can have a completely different, non-linear (even non-monotonic) intensity relationship, which mutual information — based on the joint histogram, not on any assumed functional form — can still measure correctly.</details>
2. What two numbers does the Kabsch algorithm estimate, and what closed-form tool does it use internally?
   <details><summary>Answer</summary>A rotation matrix R and a translation vector t; it uses the SVD of the cross-covariance matrix of the two (centred) point sets.</details>
3. Why can just a few outlier correspondences meaningfully bias a least-squares point-set registration?
   <details><summary>Answer</summary>Least squares minimises the SUM of squared residuals over all points, so even a few points with very large residuals contribute disproportionately to that sum and pull the fitted transform away from the majority's true alignment.</details>
4. What is the practical difference between a rigid and an affine transform, and when would you need the latter?
   <details><summary>Answer</summary>A rigid transform (rotation + translation) preserves all distances and angles; an affine transform additionally allows scaling and shear. You need affine when the imaging process itself introduces, e.g., anisotropic magnification or a slight shear, not just a rotation/translation of a perfectly rigid sample.</details>

## 🧠 Questions & discussion

- The multi-modal demo in Section 5 used contrast inversion as a stand-in for "different modality." Real multi-modal pairs (e.g. T1/T2 MRI, or fluorescence/EM) typically have a *more complex* intensity relationship than simple inversion. Would you expect mutual information's advantage over NCC to grow or shrink as the relationship gets more complex, and why?
- Section 4 showed that 3 outliers out of 15 landmarks noticeably bias Kabsch registration. Sketch (in words) how RANSAC's "random subset, then consensus" strategy would behave on this exact dataset, and why a small random subset is actually an advantage rather than a weakness here.
- Why might you prefer a point-based (landmark) registration over an intensity-based one even when both are technically available for a given pair of images?

## 🚀 Bonus — going further

- Implement a minimal RANSAC loop around `estimate_rigid_transform`: repeatedly fit on a random 3-point subset of `src_points`/`dst_with_outliers`, count inliers (residual below some tolerance) for the full set, and keep the fit with the most inliers. Confirm it recovers the true angle despite the 3 injected outliers.
- Replace the brute-force grid search in Section 5 with a coarse-to-fine search: first a wide, coarse grid, then a narrow, fine grid centred on the coarse optimum. Compare the number of `warp_rigid` evaluations needed for similar accuracy.
- Extend `warp_rigid` to an affine transform (allow independent x/y scaling and a shear parameter) and repeat the Section 4 point-based recovery experiment with an affine, rather than purely rigid, ground truth.

## 📝 Lab note

```text
Module 15 — Image registration
Data: SIMULATED fixed/moving pair, course_toolkit.simulate_registration_pair(seed=15)
True transform: angle=12.0 deg, translation=(8.0, -5.0)

Point-based (Kabsch): recovered angle = [ ]  deg, translation = ( [ ], [ ] )
Intensity-based, mono-modal (NCC): recovered angle = [ ]  deg
Intensity-based, contrast-inverted — NCC: [ ]   MI: [ ]
Notes:
```

## 📚 References

1. Zitová, B., & Flusser, J. (2003). Image registration methods: a survey. *Image and Vision Computing*.
2. Kabsch, W. (1976). A solution for the best rotation to relate two sets of vectors. *Acta Crystallographica*.
3. Viola, P., & Wells, W. M. (1997). Alignment by maximization of mutual information. *International Journal of Computer Vision*.
4. Maes, F., Collignon, A., Vandermeulen, D., Marchal, G., & Suetens, P. (1997). Multimodality image registration by maximization of mutual information. *IEEE Transactions on Medical Imaging*.
5. Fischler, M. A., & Bolles, R. C. (1981). Random sample consensus (RANSAC). *Communications of the ACM*.
6. Rueckert, D. et al. (1999). Nonrigid registration using free-form deformations: application to breast MR images. *IEEE Transactions on Medical Imaging*.
7. Sotiras, A., Davatzikos, C., & Paragios, N. (2013). Deformable medical image registration: a survey. *IEEE Transactions on Medical Imaging*.
8. Besl, P. J., & McKay, N. D. (1992). A method for registration of 3-D shapes (ICP). *IEEE Transactions on Pattern Analysis and Machine Intelligence*.

## ➡️ Next

[**Module 16 — Denoising and super-resolution**](16_denoising_and_super_resolution.ipynb): registration assumed we could already see the structures we wanted to align; next we tackle the opposite problem — recovering a clean, sharp signal from a noisy, low-resolution one.